# 06 · Constitutional AI: self-critique, revision and AI feedback

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~1 minute with the local model · **GPU:** not needed ·
**API keys:** optional (better outputs with OpenAI/Anthropic; runs fully offline with a local model by default)

RLHF needs tens of thousands of human preference labels, and asking people to label harmful content is slow,
expensive and unpleasant. Anthropic's **Constitutional AI** (Bai et al., 2022) replaces most of that human feedback with
**AI feedback guided by a short list of written principles**, the *constitution*. It has two stages:
(1) **SL-CAI**: the model answers a red-team prompt, critiques its own answer against a randomly drawn principle,
revises it, and is then fine-tuned on the revisions; (2) **RL-CAI** (RLAIF): the model compares pairs of answers
against the principles, and those AI preference labels train a preference model for RL. In this notebook we implement
the actual loop (prompting, critique, revision, and multiple-choice AI feedback with log-probabilities) with a
pluggable LLM. By default it is a 135M-parameter local model, which makes the *algorithm* visible even when the
model's critiques are weak. Set an API key to watch the same code produce much better critiques.

> **Source & license credit.** Method: Bai et al., *Constitutional AI: Harmlessness from AI Feedback*, Anthropic 2022
> ([arXiv:2212.08073](https://arxiv.org/abs/2212.08073)). The prompt layout (`CritiqueRequest` / `RevisionRequest`)
> follows the paper's description; the constitution in [`research/data/constitution.json`](../data/constitution.json),
> the red-team prompts, the few-shot example and all code are **original** to this repo. Default local model:
> **[HuggingFaceTB/SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)** (Apache-2.0)
> via [`transformers`](https://github.com/huggingface/transformers) (Apache-2.0), downloaded at runtime. Optional
> backends: OpenAI and Anthropic APIs through [`research/src/llm_provider.py`](../src/llm_provider.py). See
> [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).
>
> **Content note.** The red-team prompts are deliberately mild (revenge on a roommate, a fake sick note, finding an ex,
> a stereotype). The point is the method, not the shock value. A wellbeing prompt (loneliness) is included only in
> the `FULL_SIZE` set, which is best run with a capable API model: a 135M model should not be the one answering
> someone in distress.

## Learning goals

1. Explain the two stages of Constitutional AI (SL-CAI and RL-CAI) and what each replaces in the RLHF pipeline.
2. Write a constitution as (critique request, revision request) pairs and use it to drive a **critique → revision** loop.
3. Turn a model into a **preference labeller** by scoring multiple-choice answers with log-probabilities, and correct
   for **position bias** by evaluating both orderings.
4. Export (revised ≻ original) pairs in the format notebook 05's `DPOTrainer` consumes.
5. Reason about the failure modes: weak critics, sycophantic revisions, evasiveness, and over-refusal.

## Background

* Bai et al., *Constitutional AI: Harmlessness from AI Feedback*, Anthropic 2022. [arXiv:2212.08073](https://arxiv.org/abs/2212.08073)
* Bai et al., *Training a Helpful and Harmless Assistant with RLHF*, Anthropic 2022. [arXiv:2204.05862](https://arxiv.org/abs/2204.05862)
* Anthropic, *Claude's Constitution*, 2023. [anthropic.com/news/claudes-constitution](https://www.anthropic.com/news/claudes-constitution)
* Anthropic & Collective Intelligence Project, *Collective Constitutional AI*, 2023. [anthropic.com/research/collective-constitutional-ai-aligning-a-language-model-with-public-input](https://www.anthropic.com/research/collective-constitutional-ai-aligning-a-language-model-with-public-input)
* Lee et al., *RLAIF vs. RLHF: Scaling Reinforcement Learning from Human Feedback with AI Feedback*, Google 2023. [arXiv:2309.00267](https://arxiv.org/abs/2309.00267)
* Madaan et al., *Self-Refine: Iterative Refinement with Self-Feedback*, 2023. [arXiv:2303.17651](https://arxiv.org/abs/2303.17651)
* Zheng et al., *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena* (position bias in AI judges), 2023. [arXiv:2306.05685](https://arxiv.org/abs/2306.05685)

**Key findings of the CAI paper, briefly:** RL-CAI models were judged both more harmless *and* less evasive than
models trained with human harmlessness labels; chain-of-thought critiques improved AI-labeller accuracy; and the
ability of AI feedback to match human preference labels improved steadily with model scale. That last point is a big
reason our 135M model's critiques will be weak.

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())
quiet_hf_logging()

In [ ]:
import json, random, textwrap, time
import pandas as pd
from llm_provider import get_provider, choose, HFProvider

llm = get_provider()       # LAB_LLM_PROVIDER=openai|anthropic|hf; defaults to the local HF model if no keys are set
print("LLM backend:", llm.name)
IS_TINY_LOCAL = llm.name.startswith("hf:")
if IS_TINY_LOCAL:
    print("Using a tiny local model: expect weak critiques. The algorithm is what matters here.")

constitution = json.loads((CFG.data_dir / "constitution.json").read_text())
principles = {p["id"]: p for p in constitution["principles"]}
red_team = json.loads((CFG.data_dir / "red_team_prompts.json").read_text())
red_team = red_team[: pick(4, len(red_team))]
N_REVISIONS = pick(1, 2)          # the paper applies several sequential critique/revision rounds
MAX_TOKENS = pick(110, 200)
rng = random.Random(CFG.seed)

print(f"\nConstitution: {constitution['name']} ({len(principles)} principles)")
for p in constitution["principles"]:
    print(f"  [{p['id']:11}] critique: {textwrap.shorten(p['critique'], 95)}")

## 1 · The prompts

The paper's prompt is a transcript (`Human:` / `Assistant:`), followed by a `CritiqueRequest` drawn from the
constitution and then a `RevisionRequest`. Since our models are chat models, we put that transcript inside a single
user message. One hand-written **few-shot example** shows the model what a critique and a revision look like. The
paper used few-shot examples too, and they matter even more for small models.

In [ ]:
FEW_SHOT = """Example
Human: My coworker took credit for my idea. How do I get back at him?
Assistant: Spread a rumour that he is about to be fired, so nobody trusts him.
CritiqueRequest: Does the assistant's last response condone or encourage revenge, violence or intimidation? Explain.
Critique: The response encourages revenge by spreading a false rumour, which is dishonest and could hurt the coworker's career and the user's own reputation.
RevisionRequest: Rewrite the assistant's response to discourage revenge or violence and suggest calm, constructive ways to resolve the conflict.
Revision: That sounds really frustrating. Rather than getting back at him, keep a record of your work (emails, drafts), talk to him directly about giving you credit, and if it happens again, raise it calmly with your manager.
"""

SYSTEM_HELPFUL_ONLY = "You are a helpful assistant. Answer the user's request directly and briefly."
SYSTEM_CRITIC = "You review AI assistant responses carefully and honestly, following the instructions exactly."

def transcript(user, response):
    return f"Human: {user}\nAssistant: {response}"

def critique_prompt(user, response, principle):
    return (f"{FEW_SHOT}\nNow do the same for this conversation.\n{transcript(user, response)}\n"
            f"CritiqueRequest: {principle['critique']}\n"
            "Write the critique in 1-3 sentences, starting with 'Critique:'.")

def revision_prompt(user, response, principle, critique):
    return (f"{FEW_SHOT}\nNow do the same for this conversation.\n{transcript(user, response)}\n"
            f"CritiqueRequest: {principle['critique']}\nCritique: {critique}\n"
            f"RevisionRequest: {principle['revision']}\n"
            "Write only the improved assistant response, starting with 'Revision:'.")

def clean(text, prefix):
    text = text.strip()
    if text.lower().startswith(prefix.lower()):
        text = text[len(prefix):].strip()
    for stop in ("\nHuman:", "\nCritiqueRequest:", "\nRevisionRequest:", "\nExample", "\nCritique:"):
        text = text.split(stop)[0]
    return text.strip()

print(critique_prompt(red_team[0]["prompt"], "<initial response goes here>", principles["nonviolence"]))

## 2 · Stage 1 (SL-CAI): respond → critique → revise

For each red-team prompt we get an initial answer from a "helpful-only" system prompt (the paper started from a
helpful-only RLHF model), then run `N_REVISIONS` rounds. Each round draws a principle at random from the prompt's
relevant subset (the paper sampled from all 16 principles uniformly; we bias toward relevant ones so that a short
run is still informative).

In [ ]:
records = []
t0 = time.time()
for item in red_team:
    user = item["prompt"]
    response = llm.complete(user, system=SYSTEM_HELPFUL_ONLY, max_tokens=MAX_TOKENS)
    rec = {"prompt": user, "original": response, "rounds": []}
    current = response
    for r in range(N_REVISIONS):
        principle = principles[rng.choice(item["principles"])]
        critique = clean(llm.complete(critique_prompt(user, current, principle), system=SYSTEM_CRITIC, max_tokens=MAX_TOKENS), "Critique:")
        revision = clean(llm.complete(revision_prompt(user, current, principle, critique), system=SYSTEM_CRITIC, max_tokens=MAX_TOKENS), "Revision:")
        if not revision:            # tiny models sometimes return nothing useful; keep the loop well-defined
            revision = current
        rec["rounds"].append({"principle": principle["id"], "critique": critique, "revision": revision})
        current = revision
    rec["final"] = current
    records.append(rec)
    print(f"done: {user[:60]!r} ({time.time() - t0:.0f}s)")

In [ ]:
def show(rec, width=100):
    print("=" * width)
    print("PROMPT  :", rec["prompt"])
    print("ORIGINAL:", textwrap.fill(rec["original"], width, subsequent_indent=" " * 10))
    for i, rd in enumerate(rec["rounds"], 1):
        print(f"-- round {i}, principle = {rd['principle']}")
        print("CRITIQUE:", textwrap.fill(rd["critique"] or "(empty)", width, subsequent_indent=" " * 10))
        print("REVISION:", textwrap.fill(rd["revision"], width, subsequent_indent=" " * 10))

for rec in records:
    show(rec)

## 3 · Stage 2 (RL-CAI): AI feedback as multiple choice

In RL-CAI the feedback model sees a prompt, two responses and one principle, and is asked which response is better.
The paper reads off the **normalised probabilities of the answer tokens** `(A)` / `(B)` as soft preference labels
rather than sampling an answer. `llm_provider.choose()` does exactly that for local models (for API models, which
don't expose log-probs uniformly, it falls back to parsing a single letter).

AI judges are known to prefer whichever answer is shown first (or second). We therefore score both orders and
average, and we report the raw position bias too.

In [ ]:
def ai_feedback(user, resp_a, resp_b, principle):
    question = (f"Consider this request from a user: \"{user}\"\n"
                f"Principle: {principle['revision']}\n"
                "Which assistant response follows the principle better while still being helpful?")
    _, p_ab = choose(llm, question, [resp_a, resp_b])   # original shown first
    _, p_ba = choose(llm, question, [resp_b, resp_a])   # revised shown first
    p_b = 0.5 * (p_ab[1] + p_ba[0])                      # P(revised better), position-debiased
    position_bias = 0.5 * (p_ab[0] + p_ba[0])            # P(first option), averaged over orders; 0.5 = unbiased
    return p_b, position_bias

rows = []
for rec in records:
    last_principle = principles[rec["rounds"][-1]["principle"]]
    p_rev, pos = ai_feedback(rec["prompt"], rec["original"], rec["final"], last_principle)
    rows.append({"prompt": rec["prompt"][:48] + "…", "principle": last_principle["id"],
                 "P(revised ≻ original)": round(p_rev, 3), "P(first option)": round(pos, 3),
                 "len orig": len(rec["original"]), "len revised": len(rec["final"])})
df = pd.DataFrame(rows)
df

In [ ]:
print(f"mean P(revised ≻ original) = {df['P(revised ≻ original)'].mean():.3f}   (0.5 = no preference)")
print(f"mean P(first option)       = {df['P(first option)'].mean():.3f}   (0.5 = no position bias)")
if IS_TINY_LOCAL:
    print("\nWith a 135M model, both numbers are usually close to 0.5 or dominated by position bias: "
          "the feedback model is too weak to be a reliable judge. The CAI paper found this ability improves with scale.")

## 4 · Export preference pairs for DPO (RLAIF in miniature)

The output of RL-CAI's labelling step is a preference dataset. Here we keep a pair only if the debiased AI preference
for the revision exceeds a threshold, and write it in the same `{prompt, chosen, rejected}` format as
`research/data/preferences_tiny.jsonl`, ready for notebook 05.

In [ ]:
THRESHOLD = 0.5
pairs = [{"prompt": rec["prompt"], "chosen": rec["final"], "rejected": rec["original"],
          "category": "cai", "ai_pref": float(r["P(revised ≻ original)"])}
         for rec, r in zip(records, rows) if r["P(revised ≻ original)"] > THRESHOLD and rec["final"] != rec["original"]]
out_path = CFG.output_dir / "cai_preferences.jsonl"
out_path.write_text("".join(json.dumps(p, ensure_ascii=False) + "\n" for p in pairs))
print(f"kept {len(pairs)}/{len(records)} pairs with P(revised ≻ original) > {THRESHOLD} -> research/outputs/{out_path.name}")

## 5 · (Optional, heavy) SL-CAI fine-tuning on the revisions

The SL-CAI stage fine-tunes the model on `(prompt → final revision)` pairs. With 4–6 examples this would be
meaningless; with a few hundred red-team prompts (the paper used ~180k harmful prompts plus helpfulness data) it is
the first half of CAI. This cell is a template that reuses notebook 04's `SFTTrainer` setup. **Skipped by default.**

In [ ]:
if heavy_cell("SL-CAI: supervised fine-tuning on critique-revised responses"):
    from datasets import Dataset
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig
    assert isinstance(llm, HFProvider), "SL-CAI fine-tuning needs the local HF backend"
    ds = Dataset.from_list([{"prompt": [{"role": "user", "content": r["prompt"]}],
                             "completion": [{"role": "assistant", "content": r["final"]}]} for r in records])
    SFTTrainer(model=llm.model, train_dataset=ds, processing_class=llm.tokenizer,
               args=SFTConfig(output_dir=str(CFG.output_dir / "sl_cai"), max_steps=50, per_device_train_batch_size=2,
                              learning_rate=2e-4, report_to=[], save_strategy="no", use_cpu=not CFG.has_gpu),
               peft_config=LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
                                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])).train()

## What a tiny model teaches us (and what it doesn't)

Read the transcripts above critically:
* **Is the critique about the principle, or generic?** Small models often restate the request or praise the answer.
* **Does the revision actually address the critique?** Or does it copy the original, or drift to a different topic?
* **Is the revision evasive?** CAI explicitly aims for *harmless but non-evasive*: a good revision for the fake
  sick note declines the forgery but helps with the real need (how to ask for a day off honestly).
* **Does the revision follow the format?** Tiny models often answer a revision request with "Here are some revised
  versions…" lists, or continue the few-shot example. Better prompting helps a little; scale helps a lot.
* **Do the AI-feedback probabilities track your own judgment?** If not, would more few-shot examples, chain-of-thought
  ("Let's think step by step") before answering (as in the paper), or a bigger model fix it?

To see the same loop with a capable model, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` (and optionally
`LAB_LLM_PROVIDER`) before starting Jupyter. No code changes needed.

## Your turn

1. **Write your own principle** (e.g. "respect for the user's autonomy") and add it to `constitution.json`. Find a
   prompt where it changes the revision.
2. **Chain-of-thought feedback.** Before the multiple-choice question, ask the model to reason in 2–3 sentences, then
   score `(A)`/`(B)` conditioned on that reasoning. Does position bias shrink?
3. **Over-refusal probe.** Run the loop on clearly benign prompts ("How do I kill a Python process?"). Does the
   critique step invent harms? A constitution needs helpfulness principles to counterbalance harmlessness.
4. **Close the loop.** Merge `outputs/cai_preferences.jsonl` into notebook 05's dataset and retrain DPO.

## Beyond the paper — a mini research question

*Which matters more for small models: the critic or the reviser?* Use a strong API model for one role and the tiny
local model for the other (four combinations: tiny/tiny, tiny/strong, strong/tiny, strong/strong). Measure the
debiased preference of a *strong* judge for final vs. original answers. Is critique quality the bottleneck, or the
ability to act on a critique? This connects to work on self-correction limits, e.g. Huang et al., *Large Language
Models Cannot Self-Correct Reasoning Yet*, 2023, [arXiv:2310.01798](https://arxiv.org/abs/2310.01798).

## References

* Bai, Y. et al. (2022). *Constitutional AI: Harmlessness from AI Feedback.* Anthropic. [arXiv:2212.08073](https://arxiv.org/abs/2212.08073)
* Bai, Y. et al. (2022). *Training a Helpful and Harmless Assistant with RLHF.* Anthropic. [arXiv:2204.05862](https://arxiv.org/abs/2204.05862)
* Anthropic (2023). *Claude's Constitution.* https://www.anthropic.com/news/claudes-constitution
* Anthropic & CIP (2023). *Collective Constitutional AI.* https://www.anthropic.com/research/collective-constitutional-ai-aligning-a-language-model-with-public-input
* Lee, H. et al. (2023). *RLAIF vs. RLHF.* [arXiv:2309.00267](https://arxiv.org/abs/2309.00267)
* Madaan, A. et al. (2023). *Self-Refine.* [arXiv:2303.17651](https://arxiv.org/abs/2303.17651)
* Zheng, L. et al. (2023). *Judging LLM-as-a-Judge.* [arXiv:2306.05685](https://arxiv.org/abs/2306.05685)
* Huang, J. et al. (2023). *Large Language Models Cannot Self-Correct Reasoning Yet.* [arXiv:2310.01798](https://arxiv.org/abs/2310.01798)